# Analise inicial da base de intoxicacao exogena em idosos

Este notebook considera apenas o periodo de `2007` a `2025`, descarta o registro de `1991` e completa anos ausentes com `0` notificacoes por estado e sexo.

## Imports

In [ ]:
import sys
from pathlib import Path

from IPython.display import Image, Markdown, display

try:
    import matplotlib.pyplot as plt
    import pandas as pd
    import seaborn as sns
except ImportError as exc:
    raise ImportError(
        "Instale pandas, matplotlib e seaborn antes de executar este notebook. "
        "Exemplo no Jupyter: %pip install pandas matplotlib seaborn"
    ) from exc

cwd = Path.cwd().resolve()
if (cwd / "utils").exists():
    ROOT = cwd
elif (cwd.parent / "utils").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Nao foi possivel localizar a raiz do projeto.")

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(style="whitegrid", context="talk")
pd.options.display.float_format = "{:,.2f}".format

## Methods

In [ ]:
from utils.intoxicacao_analysis import (
    DEFAULT_END_YEAR,
    DEFAULT_START_YEAR,
    build_descriptive_stats,
    build_insight_lines,
    build_overall_by_year,
    build_state_summary,
    build_state_year,
    build_top_groups_by_state,
    build_toxic_profile,
    build_year_totals,
    filter_year_window,
    format_insights_markdown,
    load_intoxicacao_tidy,
    save_state_sex_timeseries_charts,
    summarize_year_coverage,
    write_text_report,
)

## Params

In [ ]:
DATA_PATH = ROOT / "data" / "raw" / "Dados CFF - Intoxicação exógena .xlsx"
START_YEAR = DEFAULT_START_YEAR
END_YEAR = DEFAULT_END_YEAR
TOP_N_TOXIC_GROUPS = 8
TOP_N_GROUPS_BY_STATE = 3
ASSETS_DIR = ROOT / "notebooks" / "assets"
OUTPUT_PATH = ROOT / "notebooks" / "outputs" / "insights_iniciais_intoxicacao.md"

params = pd.Series(
    {
        "data_path": str(DATA_PATH),
        "start_year": START_YEAR,
        "end_year": END_YEAR,
        "top_n_toxic_groups": TOP_N_TOXIC_GROUPS,
        "top_n_groups_by_state": TOP_N_GROUPS_BY_STATE,
        "assets_dir": str(ASSETS_DIR),
        "output_path": str(OUTPUT_PATH),
    },
    name="value",
)
display(params.to_frame())

## Read

In [ ]:
df_raw = load_intoxicacao_tidy(DATA_PATH)

display(df_raw.head())
print(f"Linhas observadas na planilha: {len(df_raw):,}")
print(f"UFs presentes: {df_raw['state'].nunique()}")
print(f"Anos observados antes do filtro: {df_raw['year'].min()} a {df_raw['year'].max()}")

## Processing

In [ ]:
df = filter_year_window(df_raw, START_YEAR, END_YEAR)
coverage = summarize_year_coverage(df, START_YEAR, END_YEAR)
year_totals = build_year_totals(df, START_YEAR, END_YEAR)
state_summary = build_state_summary(year_totals)
descriptive_stats = build_descriptive_stats(year_totals)
overall_by_year = build_overall_by_year(year_totals)
state_year = build_state_year(year_totals)
toxic_profile = build_toxic_profile(df)
top_toxic = toxic_profile.groupby("sex_label").head(TOP_N_TOXIC_GROUPS).copy()
top_toxic["share_within_sex"] = (top_toxic["share_within_sex"] * 100).round(1)
top_group_by_state = build_top_groups_by_state(df, top_n=TOP_N_GROUPS_BY_STATE)

print(f"Linhas apos filtro {START_YEAR}-{END_YEAR}: {len(df):,}")
print(f"Combinacoes estado/sexo/ano na serie completa: {len(year_totals):,}")
print(f"Intervalo final da serie: {year_totals['year'].min()} a {year_totals['year'].max()}")

display(coverage)
display(state_summary)
display(descriptive_stats.round(2))
display(top_toxic)
display(top_group_by_state)

## Graphs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.lineplot(
    data=overall_by_year,
    x="year",
    y="total_year",
    hue="sex_label",
    marker="o",
    ax=axes[0],
)
axes[0].set_title("Notificacoes anuais por sexo")
axes[0].set_xlabel("Ano")
axes[0].set_ylabel("Notificacoes")
axes[0].legend(title="Sexo")

heatmap_data = state_year.pivot(index="state", columns="year", values="total_year").fillna(0.0)
sns.heatmap(heatmap_data, cmap="YlOrRd", linewidths=0.3, ax=axes[1])
axes[1].set_title("Intensidade anual por estado")
axes[1].set_xlabel("Ano")
axes[1].set_ylabel("Estado")

plt.tight_layout()

state_chart_paths = save_state_sex_timeseries_charts(
    year_totals=year_totals,
    output_dir=ASSETS_DIR,
    start_year=START_YEAR,
    end_year=END_YEAR,
)

state_chart_table = pd.DataFrame(
    {
        "state": sorted(year_totals["state"].unique()),
        "asset_path": [str(path.relative_to(ROOT)) for path in state_chart_paths],
    }
)
display(state_chart_table)

for row, path in zip(state_chart_table.itertuples(index=False), state_chart_paths):
    display(Markdown(f"### {row.state}"))
    display(Image(filename=str(path)))

plt.figure(figsize=(12, 7))
sns.barplot(data=top_toxic, x="count", y="toxic_group", hue="sex_label")
plt.title("Principais grupos toxicos por sexo")
plt.xlabel("Notificacoes acumuladas")
plt.ylabel("Grupo do agente toxico")
plt.tight_layout()

## Insights

In [ ]:
latest_year_table = (
    year_totals.loc[year_totals["year"] == END_YEAR]
    .pivot(index="state", columns="sex_label", values="total_year")
    .fillna(0.0)
    .sort_index()
)

filled_zero_table = coverage.loc[
    coverage["filled_zero_years"] > 0,
    ["state_code", "sex_label", "filled_zero_years", "missing_year_list"],
].reset_index(drop=True)

insight_lines = build_insight_lines(
    year_totals=year_totals,
    state_summary=state_summary,
    toxic_profile=toxic_profile,
    coverage=coverage,
    start_year=START_YEAR,
    end_year=END_YEAR,
)

display(latest_year_table)
if filled_zero_table.empty:
    display(pd.DataFrame({"observacao": ["Nenhum ano precisou ser preenchido com zero."]}))
else:
    display(filled_zero_table)

display(Markdown("### Principais achados\n" + "\n".join(f"- {line}" for line in insight_lines)))

## Write insights

In [ ]:
report_text = format_insights_markdown(
    insight_lines,
    title="Insights iniciais - intoxicacao exogena em idosos",
)
written_path = write_text_report(OUTPUT_PATH, report_text)

print(f"Arquivo gerado: {written_path}")
display(Markdown(report_text))